# Part 1 - Single Prediction

For each input of a single value of blade_angle, multiplies it by the single weight to produce a single output number as prediction.

In [ ]:
# week2/part1_single_prediction.py
blade_angle = [8.5, 9.5, 9.9, 9.0]
weight = 0.5

# Setting the weight to 0.2 over 0.5 would lower each of the values within blade_angle,
# since it is the only modifier here with the initial values. If it were a negative value,
# it would flip the sign of the input as well, sending it in the opposite direction.

def neural_network(input, weight, debug = False):
    """
    Inputs: input, weight
    Loop through input array, and multiply the weight passed in.
    """
    prediction = []
    for i in input:
        prediction.append(i * weight)

    if debug:
        print(prediction)
    return prediction

neural_network(blade_angle, weight, debug = True)


[4.25, 4.75, 4.95, 4.5]

# Part 2 - Multiple Inputs

For each input of three values, blade_angle, balance and breath, multiplies each one by their assigned weight and sums them to produce a single output number as prediction.

# Part 3 - Multiple Outputs

For each input of 1 value, blade_angle, multiplies it by three separate assigned weights, yielding 3 separate output numbers.

# Part 4 - Multiple Inputs, Multiple Outputs

For each of three neurons, when sent an input of three values, blade_angle, balance and breath, finds the dot product of that neuron's weights and the input vector.

In [9]:
from part2_multiple_inputs import w_sum

weights = [
    # angle balance breath
    [0.1, 0.1, -0.3],  # -> opens_left?
    [0.1, 0.2, 0.0],  # -> strikes_high?
    [0.0, 1.3, 0.1],  # -> feints?
]

blade_angle = [8.5, 9.5, 9.9, 9.0]  # degrees off-vertical
balance = [0.65, 0.80, 0.80, 0.90]  # balance reading
breath = [1.2, 1.3, 0.5, 1.0]  # exhalations per second
input = [blade_angle[0], balance[0], breath[0]]
# Expected for sensing 0 (rounded): [0.555, 0.98, 0.965]

# Bundle each sensing
# [blade_angle[a], balance[a], breath[a]]
# together in an array of sensings
sensings = list(zip(blade_angle, balance, breath))


# One-line list-comprehension scratch version
def vect_mat_mul(vect, matrix):
    """Takes a vector of inputs, and uses the weighted
    sum of that vector and each row of the matrix for
    a list of outputs."""
    return [w_sum(vect, row, debug=False) for row in matrix]


# There should be four sparring sensings
def scratch_nn(debug=False):
    """Runs our scratch implementation of multi-in,
    multi-out neural network on all four sensings"""
    print("-- From scratch --")
    if debug:
        print(f"There are {len(sensings)} sensings: {sensings}")
    outputs = [0] * len(sensings)
    for i in range(len(sensings)):
        outputs[i] = vect_mat_mul(sensings[i], weights)
        print(outputs[i])
    return outputs


# Preconditions: input and weights are NumPy arrays
# and the numpy module is imported
def understand_np_shapes(input, weights):
    """Prints and explains the shapes of a sensing
    (row of input) and the weight matrix"""
    print()
    print("-- NumPy version --")
    print("Understanding Shapes...")
    print(f"Shape of any row of inputs: {input.shape}")
    print(f"Shape of weight matrix: {weights.shape}")
    print(
        "A shape of (3,) means that the array is one-dimensional and has three elements."
    )
    print(
        "A shape of (3,3) means that we have a square matrix with three rows and three columns, or elements in each row."
    )


# -- NumPy version --


# Precondition: numpy is imported
def np_nn(inputs, weights):
    print()
    print("Vector Matrix Multiplication with NumPy...")
    outputs = [0] * len(inputs)
    for i in range(len(inputs)):
        outputs[i] = inputs[i].dot(weights.T)
        print(outputs[i])
    return outputs


def main():
    # Get scratch results
    scratch_outputs = scratch_nn(debug=True)

    # -- NumPy version --
    import numpy as np

    # Convert inputs and weights to NumPy arrays
    np_weights = np.asarray(weights)
    np_sensings = np.asarray(sensings)

    # Print and explain shapes
    understand_np_shapes(np_sensings[0], np_weights)

    # Get NumPy results
    np_outputs = np_nn(np_sensings, np_weights)

    # Compare outputs for closeness
    for i in range(len(scratch_outputs)):
        print(
            f"Sensing {i}: {'close enough' if np.allclose(scratch_outputs[i], np_outputs[i], rtol=1e-9, atol=1e-9) else 'not close enough'}"
        )


if __name__ == "__main__":
    main()


-- From scratch --
There are 4 sensings: [(8.5, 0.65, 1.2), (9.5, 0.8, 1.3), (9.9, 0.8, 0.5), (9.0, 0.9, 1.0)]
[0.555, 0.9800000000000001, 0.9650000000000001]
[0.64, 1.11, 1.17]
[0.92, 1.1500000000000001, 1.09]
[0.69, 1.08, 1.2700000000000002]

-- NumPy version --
Understanding Shapes...
Shape of any row of inputs: (3,)
Shape of weight matrix: (3, 3)
A shape of (3,) means that the array is one-dimensional and has three elements.
A shape of (3,3) means that we have a square matrix with three rows and three columns, or elements in each row.

Vector Matrix Multiplication with NumPy...
[0.555 0.98  0.965]
[0.64 1.11 1.17]
[0.92 1.15 1.09]
[0.69 1.08 1.27]
Sensing 0: close enough
Sensing 1: close enough
Sensing 2: close enough
Sensing 3: close enough


# Part 5 - Hidden Layers

Runs the code of part 4 twice, first on the input to get a second set of 3 outputs, then piping that back into another set of weights, giving a final output of 3 values.

In [8]:
import numpy as np
from part4_multi_in_multi_out import vect_mat_mul
from part4_multi_in_multi_out import np_nn


# angle balance breath
ih_wgt = [[0.1, 0.2, -0.1], # -> hid[0]
[-0.1, 0.1, 0.9], # -> hid[1]
[0.1, 0.4, 0.1]] # -> hid[2]

# hid0 hid1 hid2
hp_wgt = [[0.3, 1.1, -0.3], # -> opens_left?
[0.1, 0.2, 0.0], # -> strikes_high?
[0.0, 1.3, 0.1]] # -> feints?


### a bunch of little attempts to replicate stuff myself; 
## this way I understand what it's doing when I use others' code
## elementwise multiply each value in 2 equal-length vectors, then sum
#def dot_prod(vec1, vec2):
#    # check they're the same length
#    assert len(vec1) == len(vec2)
#    # start at sum = 0
#    output = 0
#    # sequentially add the element-wise products of the vectors' values
#    for i in range(len(vec1)):
#        output += vec1[i] * vec2[i]
#
#    return output
#
#def matrix_vect_mult(matrix, vector):
#    # verify that we're multiplying an m x n matrix by an n x 1 vector
#    # by checking that the second dimension of the first row of the matrix
#    # is equal to the height of the vector (which we presume to be a vector)
#    # assumes the matrix is rectangular, that is, same width at all points
#    assert len(matrix[0]) == len(vector)
#
#    # generate output vector of 0s, as long as the height of the matrix
#    # final result of multiplying m x n matrix by n x 1 vector is m x 1 vector
#    output = [0]*len(matrix)
#
#    # see above re: length
#    for i in range(len(matrix)):
#        # the i'th value of the vector should be equal to
#        # the dot product of the entire vector with the i'th row
#        # of the matrix.
#        output[i] = dot_prod(matrix[i], vector)
#
#    return output



# works on an arbitrary number of equal-sized layers with the same size as the input
# e.g. can take 8 layers of weights for 3 inputs each and a vector of length 3
def neural_network(weights_sequence, input):
    # store starting values
    cur = input
    stored_layers = [0]*len(weights_sequence)
    #sequentially multiply starting values by the matrix of the weights
    #in each layer, proceeding to the next one

    
    for i in range(len(weights_sequence)):
        cur = vect_mat_mul(cur, weights_sequence[i])
        stored_layers[i] = cur

    return stored_layers


def main():
    weights = [ih_wgt, hp_wgt]
    blade_angle = [8.5, 9.5, 9.9, 9.0] # degrees off-vertical
    balance = [0.65, 0.80, 0.80, 0.90] # balance reading
    breath = [1.2, 1.3, 0.5, 1.0] # exhalations per second

    # trust that there are the same number of entries
    for i in range(len(blade_angle)):
        input = [blade_angle[i], balance[i], breath[i]]
        pred=neural_network(weights, input)
        for j in range(len(pred)):
            print(f"Sensing {i}, Layer {j+1}: {pred[j]}")

    # Expected for sensing 0 (rounded): hidden = [0.86, 0.295, 1.23]
    # pred = [0.2135, 0.145, 0.5065]

    print("Each hidden value is an intermediate stage, something that may or may not"
    " correspond to something we understand, storing some amount of information in some way."
    " But to not all collapse into one layer of summed linear transformations, we need"
    " something nonlinear (and non-polynomial!), applied to all these to mess with"
    " them such that you can get odd behavior.")

    #wait, why are we doing this with np.dot? This seems a little more manual than necessary.
    print("--Numpy Version--")
    np_weights = np.asarray(weights)
    np_sensings = np.asarray(list(zip(blade_angle, balance, breath)))


    print(f"{np_weights.shape} - a 2 x 3 x 3 vector, it contains 2 layers of 3 vectors of the 3 weights of that neuron.")
    print(f"{np_sensings.shape} - a 4 x 3 vector, it contains 4 sensings, each containing a vector made of the 3 variables blad_angle, balance and breath")
    for i in range(len(np_sensings)):
        input = [np_sensings[i]]

        np_hidden_layer = np_nn(input, np_weights[0])
        np_final_layer = np_nn(np_hidden_layer, np_weights[1])






if __name__ == "__main__":
    main()


Sensing 0, Layer 1: [0.8600000000000001, 0.29499999999999993, 1.23]
Sensing 0, Layer 2: [0.21350000000000002, 0.14500000000000002, 0.5065]
Sensing 1, Layer 1: [0.9800000000000001, 0.30000000000000004, 1.4]
Sensing 1, Layer 2: [0.20400000000000013, 0.15800000000000003, 0.53]
Sensing 2, Layer 1: [1.1, -0.46000000000000013, 1.36]
Sensing 2, Layer 2: [-0.5840000000000003, 0.017999999999999988, -0.4620000000000002]
Sensing 3, Layer 1: [0.9800000000000001, 0.08999999999999997, 1.36]
Sensing 3, Layer 2: [-0.015000000000000013, 0.11600000000000002, 0.253]
Each hidden value is an intermediate stage, something that may or may not correspond to something we understand, storing some amount of information in some way. But to not all collapse into one layer of summed linear transformations, we need something nonlinear (and non-polynomial!), applied to all these to mess with them such that you can get odd behavior.
--Numpy Version--
(2, 3, 3) - a 2 x 3 x 3 vector, it contains 2 layers of 3 vectors of